<a href="https://colab.research.google.com/github/Akhilesh7866/citizen-grievance-nlp/blob/rakshita/citizen_grievance_nlp_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1: Data Collection, Text Preprocessing & EDA
## AI-Driven Citizen Grievance & Sentiment Analysis System

**Dataset:** NYC 311 Service Requests  
**Tools:** Python, spaCy, WordCloud, Scikit-learn, Matplotlib  
**Goal:** Clean raw complaint text and perform EDA to understand common civic issues


In [2]:
# Install required libraries (run once)
# !pip install pandas spacy wordcloud tqdm
# !python -m spacy download en_core_web_sm

import pandas as pd
import re
import string
import spacy
from tqdm import tqdm
from spacy.lang.en.stop_words import STOP_WORDS
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS
from sklearn.feature_extraction.text import CountVectorizer

# Load spaCy English model
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [4]:
# Load the dataset
df = pd.read_csv("NYC311data.csv")  # adjust filename to yours

# Basic inspection
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'NYC311data.csv'

In [ ]:
# Check for nulls and data types
print(df.info())
print("\nNull counts:\n", df.isnull().sum())

## Preprocessing Strategy

We will clean the `Descriptor` column using the following steps:

| Step | Operation | Purpose |
|------|-----------|----------|
| 1 | Lowercase | Normalize text case |
| 2 | Remove URLs & emails | Eliminate noise |
| 3 | Remove special characters & digits | Keep alphabetic text only |
| 4 | Tokenize & remove stopwords | Remove low-value words |
| 5 | Lemmatize tokens | Reduce words to base form |
| 6 | Reconstruct cleaned text | Save as `cleaned_text` column |

In [ ]:
TEXT_COLUMN = "Descriptor"

def clean_text_basic(text):
    """Steps 1-3: fast regex cleaning (no spaCy yet)"""
    if not isinstance(text, str):
        return ""
    text = text.lower()                                   # Step 1: Lowercase
    text = re.sub(r"http\S+|www\S+", "", text)           # Step 2: Remove URLs
    text = re.sub(r"\S+@\S+", "", text)                  # Step 2: Remove emails
    text = re.sub(r"[^a-z\s]", " ", text)                # Step 3: Remove special chars
    text = re.sub(r"\s+", " ", text).strip()             # Step 3: Normalize whitespace
    return text

# Step 1-3: fast regex pass
print("Applying regex cleaning...")
df["temp_text"] = df[TEXT_COLUMN].apply(clean_text_basic)

# Step 4-5: batch lemmatization with progress bar
print("Applying lemmatization (batch mode)...")
cleaned_texts = []

for doc in tqdm(nlp.pipe(df["temp_text"], batch_size=500),
                total=len(df), desc="Lemmatizing"):
    tokens = [
        token.lemma_                    # Step 5: Lemmatize
        for token in doc
        if not token.is_stop            # Step 4: Remove stopwords
        and not token.is_punct
        and len(token.text) > 2
    ]
    cleaned_texts.append(" ".join(tokens))  # FIX: space separator prevents word merging

df["cleaned_text"] = cleaned_texts
df.drop(columns=["temp_text"], inplace=True)

print(f"Done! Cleaned {len(df):,} rows.")
df[["Descriptor", "cleaned_text"]].head(5)

In [ ]:
# Count empty rows after cleaning (may indicate data issues)
empty_count = df["cleaned_text"].str.strip().eq("").sum()
print(f"Empty rows after cleaning: {empty_count} / {len(df)}")

# Check average token count
df["token_count"] = df["cleaned_text"].apply(lambda x: len(x.split()))
print("\nToken count stats:")
print(df["token_count"].describe())

In [ ]:
print(f"Total rows: {len(df)}")

## Preprocessing Summary

| Step | Action | Result |
|------|--------|--------|
| Lowercase | `str.lower()` | Normalized case |
| URL/email removal | regex | Removed noise |
| Special char removal | `regex [^a-z\s]` | Clean alphabet only |
| Stopword removal | spaCy `is_stop` | ~30–50% token reduction |
| Lemmatization | spaCy `token.lemma_` | Base word forms |

Output: `cleaned_text` column — ready for Word Cloud and N-gram analysis.

In [ ]:
# Use cleaned text for word cloud
text = " ".join(df["cleaned_text"].dropna().astype(str))

In [ ]:
stopwords = set(STOPWORDS)

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color="white",
    stopwords=stopwords,
    collocations=False,
    colormap="Blues"
).generate(text)

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud — Most Common Terms in NYC 311 Complaints",
          fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

Word Cloud Analysis

The word cloud visualization highlights the most frequently occurring terms in citizen complaints. Larger words represent higher frequency. From the visualization, common issues such as music parties, loud noise, illegal parking, and access blockage appear frequently, indicating the major concerns reported by citizens.

In [ ]:
# Word cloud per complaint type
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Word Cloud by Top 4 Complaint Types", fontsize=14, fontweight="bold")
axes = axes.flatten()

top_types = df["Complaint Type"].value_counts().head(4).index

for ax, complaint_type in zip(axes, top_types):
    subset_text = " ".join(
        df[df["Complaint Type"] == complaint_type]["cleaned_text"].dropna()
    )
    wc = WordCloud(width=400, height=300, background_color="white",
                   stopwords=stopwords, collocations=False).generate(subset_text)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(complaint_type, fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
text_data = df["cleaned_text"].dropna()

In [ ]:
##Function to Calculate N-grams
def get_top_ngrams(corpus, ngram_range=(1,1), top_n=20):

    vectorizer = CountVectorizer(ngram_range=ngram_range, stop_words='english')

    X = vectorizer.fit_transform(corpus)

    sum_words = X.sum(axis=0)

    words_freq = [(word, sum_words[0, idx])
                  for word, idx in vectorizer.vocabulary_.items()]

    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

    return words_freq[:top_n]

In [ ]:
##Unigram Frequency (Single Words)
top_unigrams = get_top_ngrams(text_data, (1,1), 20)

unigram_df = pd.DataFrame(top_unigrams, columns=["Word","Frequency"])

unigram_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(unigram_df["Word"], unigram_df["Frequency"], color="#2563EB", alpha=0.85)
plt.gca().invert_yaxis()
plt.title("Top 20 Unigrams in Citizen Complaints", fontsize=13, fontweight="bold")
plt.xlabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
##Bigram Frequency (Two Word Phrases)
top_bigrams = get_top_ngrams(text_data, (2,2), 20)

bigram_df = pd.DataFrame(top_bigrams, columns=["Bigram","Frequency"])

bigram_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(bigram_df["Bigram"], bigram_df["Frequency"], color="#F59E0B", alpha=0.85)
plt.gca().invert_yaxis()
plt.title("Top 20 Bigrams in Citizen Complaints", fontsize=13, fontweight="bold")
plt.xlabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
##Trigram Frequency (Three Word Phrases)
top_trigrams = get_top_ngrams(text_data, (3,3), 20)

trigram_df = pd.DataFrame(top_trigrams, columns=["Trigram","Frequency"])

trigram_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(trigram_df["Trigram"], trigram_df["Frequency"], color="#10B981", alpha=0.85)
plt.gca().invert_yaxis()
plt.title("Top 20 Trigrams in Citizen Complaints", fontsize=13, fontweight="bold")
plt.xlabel("Frequency")
plt.tight_layout()
plt.show()

## N-gram Frequency Analysis

N-grams are groups of words used to find the most common words and phrases in citizen complaints.

- **Unigrams** show the most common single words in complaints.
- **Bigrams** show common two-word phrases like *illegal parking* or *loud music*.
- **Trigrams** show three-word phrases that give more context, such as *loud music party*.

This analysis helps understand the main problems reported by citizens, so government departments can identify important issues and respond faster.

# Week 2: Topic Modeling & Department Categorization

## AI-Driven Citizen Grievance & Sentiment Analysis System

**Dataset:** NYC 311 Service Requests  
**Tools:** Python, Scikit-learn, TF-IDF, Logistic Regression, Random Forest  
**Goal:** Convert cleaned text into numerical features using TF-IDF and build a machine learning model to classify complaints into departments.

## TF-IDF Feature Extraction

TF-IDF (Term Frequency - Inverse Document Frequency) converts text data into numerical form so that machine learning models can understand it.

It gives higher importance to important words and reduces the weight of common words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Use cleaned text
X = df["cleaned_text"].dropna()

# Target column (department / complaint type)
y = df["Complaint Type"].loc[X.index]

print("Text samples:", X.shape)
print("Labels:", y.shape)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_tfidf = tfidf.fit_transform(X)

print("TF-IDF Matrix Shape:", X_tfidf.shape)

In [ ]:
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

In [ ]:
import numpy as np

# Average score of each word
scores = np.mean(X_tfidf.toarray(), axis=0)

# Top 20 words
top_indices = scores.argsort()[-20:][::-1]

top_words = [(tfidf.get_feature_names_out()[i], scores[i]) for i in top_indices]

top_words

In [ ]:
words = [w[0] for w in top_words]
values = [w[1] for w in top_words]

import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.barh(words, values)
plt.gca().invert_yaxis()
plt.title("Top TF-IDF Important Words")
plt.xlabel("Score")
plt.show()

## TF-IDF Analysis

TF-IDF helps identify the most important words in citizen complaints.

This representation will be used for training machine learning models for complaint classification.

In [6]:
# Step 1: Import required libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


# Step 2: Use TF-IDF features and labels
X_features = X_tfidf
y_labels = y


# Step 3: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=42
)


# Step 4: Create Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)


# Step 5: Train model
rf_model.fit(X_train, y_train)


# Step 6: Predict
y_pred = rf_model.predict(X_test)


# Step 7: Evaluate
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n", classification_report(y_test, y_pred))

NameError: name 'X_tfidf' is not defined